# 07 - SIMCA Mixture Application

This notebook applies the final SIMCA models selected in notebook 06B to mixture images.

It intentionally stays lightweight:

- restore full model configurations from the compact 06B selection and the 06A candidate panel;
- refit final models on pure target batches 1, 2, 3, and 4;
- project mixture images only;
- evaluate object and pixel diagnostics using the mixture position-reference truth;
- save compact output tables for a later reporting script;
- show only a few notebook-level tables and figures.


## Inputs And Outputs

Required inputs:

- `results/06B_simca_final_selection_<RESULTS_TAG>/final_selected_models.parquet`
- `results/06A_simca_pure_test_<RESULTS_TAG>/pure_test_candidate_panel.parquet`
- `results/04C_simca_concat_refit_<RESULTS_TAG>/validation_3way_selected_thresholds.parquet`
- `results/03_pca_<RESULTS_TAG>/pca_selected_preprocessings.parquet`
- `HSI Data/processed/nir_uco_database.h5`

Main outputs:

- `mixture_selected_configs.parquet`
- `mixture_metrics_long.parquet`
- `mixture_summary.parquet`
- `mixture_object_diagnostics_by_image.parquet`
- `mixture_pixel_diagnostics_by_image.parquet`
- `mixture_3way_object_diagnostics_by_image.parquet`
- `mixture_objects.parquet`
- `mixture_pixels.parquet` when enabled
- `mixture_3way_objects.parquet`


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError("Could not find project root. Run from the project root or notebooks/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

from src import experiment_config as expcfg
from src.io.database_h5 import load_nir_uco_h5
from src.spectra.band_selection import (
    select_wavelength_range_from_database,
    wavelength_selection_summary,
)
from src.utils import list_result_files
from src.workflows.simca import make_target_train_filters
from src.workflows.simca_candidates import build_pca_preprocessing_configs_by_matrix_family
from src.workflows.simca_mixture import (
    DEFAULT_MIXTURE_EVALUATION_STAGE,
    build_mixture_guardrails,
    build_mixture_projection_filters,
    build_mixture_protocol,
    choose_mixture_diagnostic_images,
    load_existing_mixture_outputs,
    missing_existing_mixture_paths,
    restore_mixture_selected_configs,
    run_mixture_application_batches,
    save_mixture_outputs,
    validate_mixture_guardrails,
    validate_mixture_outputs,
)
from src.workflows.simca_tables import read_simca_table, write_simca_table
from src.visualization.plot_model_selection import plot_model_metric_ranking
from src.visualization.plot_reporting import (
    plot_mixture_diagnostic_panel,
    plot_per_image_performance,
)


PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


## Configuration

Set `RUN_MIXTURE_REFIT=False` and `USE_EXISTING_MIXTURE_OUTPUTS=True` to reload existing outputs without recomputing the final application.

`SAVE_COMBINED_PIXEL_TABLES=True` is useful for notebook diagnostics and for the future reporting script. If memory becomes limiting, set it to `False` and use batch pixel outputs or rerun selected diagnostics only.


In [2]:
RESULTS_TAG = expcfg.DEFAULT_RESULTS_TAG
WAVELENGTH_MODE = expcfg.DEFAULT_WAVELENGTH_MODE

USE_WAVELENGTH_WINDOW = False
WINDOW_MIN_NM = 1225.0
WINDOW_MAX_NM = 1675.0

DB_H5_PATH = PROJECT_ROOT / "HSI Data" / "processed" / "nir_uco_database.h5"

RESULTS_03_DIR = PROJECT_ROOT / "results" / f"03_pca_{RESULTS_TAG}"
RESULTS_04C_DIR = PROJECT_ROOT / "results" / f"04C_simca_concat_refit_{RESULTS_TAG}"
RESULTS_06A_DIR = PROJECT_ROOT / "results" / f"06A_simca_pure_test_{RESULTS_TAG}"
RESULTS_06B_DIR = PROJECT_ROOT / "results" / f"06B_simca_final_selection_{RESULTS_TAG}"
RESULTS_DIR = PROJECT_ROOT / "results" / f"07_simca_mixture_application_{RESULTS_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_DIR = RESULTS_DIR / "mixture_batches"
BATCH_METRICS_DIR = BATCH_DIR / "metrics"
BATCH_OBJECTS_DIR = BATCH_DIR / "objects"
BATCH_PIXELS_DIR = BATCH_DIR / "pixels"
BATCH_3WAY_OBJECTS_DIR = BATCH_DIR / "objects_3way"
for directory in [BATCH_DIR, BATCH_METRICS_DIR, BATCH_OBJECTS_DIR, BATCH_PIXELS_DIR, BATCH_3WAY_OBJECTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PCA_SELECTED_PREPROCESSINGS_PATH = RESULTS_03_DIR / "pca_selected_preprocessings.parquet"
THREE_WAY_THRESHOLDS_PATH = RESULTS_04C_DIR / "validation_3way_selected_thresholds.parquet"
PURE_TEST_CANDIDATE_PANEL_PATH = RESULTS_06A_DIR / "pure_test_candidate_panel.parquet"
FINAL_SELECTED_MODELS_PATH = RESULTS_06B_DIR / "final_selected_models.parquet"

MIXTURE_PATHS = {
    "selected_configs": RESULTS_DIR / "mixture_selected_configs.parquet",
    "2way_object_metrics": RESULTS_DIR / "mixture_2way_object_metrics.parquet",
    "2way_pixel_metrics": RESULTS_DIR / "mixture_2way_pixel_metrics.parquet",
    "3way_object_metrics": RESULTS_DIR / "mixture_3way_object_metrics.parquet",
    "metrics_long": RESULTS_DIR / "mixture_metrics_long.parquet",
    "object_image_diagnostics": RESULTS_DIR / "mixture_object_diagnostics_by_image.parquet",
    "pixel_image_diagnostics": RESULTS_DIR / "mixture_pixel_diagnostics_by_image.parquet",
    "3way_object_image_diagnostics": RESULTS_DIR / "mixture_3way_object_diagnostics_by_image.parquet",
    "pixel_errors_by_image": RESULTS_DIR / "mixture_pixel_errors_by_image.parquet",
    "errors": RESULTS_DIR / "mixture_errors.parquet",
    "batch_manifest": RESULTS_DIR / "mixture_batch_manifest.parquet",
    "objects": RESULTS_DIR / "mixture_objects.parquet",
    "pixels": RESULTS_DIR / "mixture_pixels.parquet",
    "3way_objects": RESULTS_DIR / "mixture_3way_objects.parquet",
    "summary": RESULTS_DIR / "mixture_summary.parquet",
    "guardrails": RESULTS_DIR / "mixture_guardrails.parquet",
    "protocol": RESULTS_DIR / "mixture_protocol.parquet",
}

TARGET_CLASS = expcfg.TARGET_CLASS
NON_TARGET_LABEL = expcfg.NON_TARGET_LABEL
REFERENCE_CLASSES = expcfg.REFERENCE_CLASSES

MIXTURE_FINAL_TRAIN_BATCHES = list(expcfg.MIXTURE_FINAL_TRAIN_BATCHES)
MIXTURE_FINAL_TRAIN_FILTERS = make_target_train_filters(
    target_class=TARGET_CLASS,
    train_batches=MIXTURE_FINAL_TRAIN_BATCHES,
)
MIXTURE_FILTERS = build_mixture_projection_filters()

RUN_MIXTURE_REFIT = True
USE_EXISTING_MIXTURE_OUTPUTS = True
MIXTURE_BATCH_SIZE = expcfg.MIXTURE_APPLICATION_BATCH_SIZE
MAX_MODELS_PER_TRACK = expcfg.MIXTURE_APPLICATION_MAX_MODELS_PER_TRACK

SAVE_BATCH_METRIC_TABLES = expcfg.MIXTURE_APPLICATION_SAVE_BATCH_METRIC_TABLES
SAVE_BATCH_OBJECT_TABLES = expcfg.MIXTURE_APPLICATION_SAVE_BATCH_OBJECT_TABLES
SAVE_BATCH_PIXEL_TABLES = expcfg.MIXTURE_APPLICATION_SAVE_BATCH_PIXEL_TABLES
SAVE_BATCH_3WAY_OBJECT_TABLES = expcfg.MIXTURE_APPLICATION_SAVE_BATCH_3WAY_OBJECT_TABLES

SAVE_COMBINED_OBJECT_TABLES = expcfg.MIXTURE_APPLICATION_SAVE_COMBINED_OBJECT_TABLES
SAVE_COMBINED_PIXEL_TABLES = expcfg.MIXTURE_APPLICATION_SAVE_COMBINED_PIXEL_TABLES
SAVE_COMBINED_3WAY_OBJECT_TABLES = expcfg.MIXTURE_APPLICATION_SAVE_COMBINED_3WAY_OBJECT_TABLES
KEEP_ONLY_ASSIGNED_TRACK_METRICS = expcfg.MIXTURE_APPLICATION_KEEP_ONLY_ASSIGNED_TRACK_METRICS

EVALUATION_STAGE = DEFAULT_MIXTURE_EVALUATION_STAGE
RANDOM_STATE = expcfg.RANDOM_STATE
REPLACE_BALANCED_PIXELS = expcfg.REPLACE_BALANCED_PIXELS
CV_N_SPLITS = expcfg.CV_N_SPLITS
CV_GROUP_COL = expcfg.CV_GROUP_COL

print("Input 06B:", FINAL_SELECTED_MODELS_PATH)
print("Input 06A candidate panel:", PURE_TEST_CANDIDATE_PANEL_PATH)
print("Input 04C thresholds:", THREE_WAY_THRESHOLDS_PATH)
print("Output dir:", RESULTS_DIR)
print("RUN_MIXTURE_REFIT:", RUN_MIXTURE_REFIT)
print("USE_EXISTING_MIXTURE_OUTPUTS:", USE_EXISTING_MIXTURE_OUTPUTS)
print("MIXTURE_BATCH_SIZE:", MIXTURE_BATCH_SIZE)
print("SAVE_COMBINED_PIXEL_TABLES:", SAVE_COMBINED_PIXEL_TABLES)


Input 06B: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06B_simca_final_selection_non_noisy_all\final_selected_models.parquet
Input 06A candidate panel: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06A_simca_pure_test_non_noisy_all\pure_test_candidate_panel.parquet
Input 04C thresholds: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_concat_refit_non_noisy_all\validation_3way_selected_thresholds.parquet
Output dir: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\07_simca_mixture_application_non_noisy_all
RUN_MIXTURE_REFIT: True
USE_EXISTING_MIXTURE_OUTPUTS: True
MIXTURE_BATCH_SIZE: 10
SAVE_COMBINED_PIXEL_TABLES: True


## Load Database And Frozen Inputs

The selected models are compact in 06B, so the full refit parameters are restored from the 06A candidate panel. Fixed 3-way thresholds remain the thresholds selected before pure test in 04C.


In [3]:
required_paths = [
    DB_H5_PATH,
    PCA_SELECTED_PREPROCESSINGS_PATH,
    THREE_WAY_THRESHOLDS_PATH,
    PURE_TEST_CANDIDATE_PANEL_PATH,
    FINAL_SELECTED_MODELS_PATH,
]
missing_paths = [path for path in required_paths if not Path(path).exists()]
if missing_paths:
    raise FileNotFoundError("Missing required input file(s): " + ", ".join(map(str, missing_paths)))

object_db, image_db = load_nir_uco_h5(DB_H5_PATH, reconstruct_heavy_object_arrays=True)

if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=WINDOW_MIN_NM,
        max_nm=WINDOW_MAX_NM,
    )
    wavelength_selection_df = wavelength_selection_summary(wavelength_info)
else:
    first_obj = next(iter(object_db.values()))
    wavelengths = first_obj.get("wavelengths")
    wavelengths = np.asarray(wavelengths, dtype=float) if wavelengths is not None else None
    wavelength_selection_df = pd.DataFrame()

if wavelengths is None:
    raise RuntimeError("No wavelength axis found in object_db.")

pca_selected_preprocessings_df = read_simca_table(PCA_SELECTED_PREPROCESSINGS_PATH, required=True)
preprocessing_configs_by_family = build_pca_preprocessing_configs_by_matrix_family(
    pca_selected_preprocessings_df
)

final_selected_models_df = read_simca_table(FINAL_SELECTED_MODELS_PATH, required=True)
pure_test_candidate_panel_df = read_simca_table(PURE_TEST_CANDIDATE_PANEL_PATH, required=True)
three_way_thresholds_df = read_simca_table(THREE_WAY_THRESHOLDS_PATH, required=True)

selected_configs_df, selected_thresholds_df = restore_mixture_selected_configs(
    final_selected_models_df=final_selected_models_df,
    candidate_panel_df=pure_test_candidate_panel_df,
    thresholds_df=three_way_thresholds_df,
    max_models_per_track=MAX_MODELS_PER_TRACK,
)

guardrails_df = validate_mixture_guardrails(
    build_mixture_guardrails(
        selected_configs_df=selected_configs_df,
        final_selected_models_df=final_selected_models_df,
        candidate_panel_df=pure_test_candidate_panel_df,
        thresholds_df=selected_thresholds_df,
        object_db=object_db,
        train_batches=MIXTURE_FINAL_TRAIN_BATCHES,
        projection_filters=MIXTURE_FILTERS,
        expected_tracks=expcfg.SIMCA_SELECTION_TRACKS,
        target_class=TARGET_CLASS,
    )
)
write_simca_table(guardrails_df, MIXTURE_PATHS["guardrails"])
write_simca_table(selected_configs_df, MIXTURE_PATHS["selected_configs"])

print("Images:", len(image_db))
print("Objects:", len(object_db))
print("Final selected rows:", final_selected_models_df.shape)
print("Restored selected configs:", selected_configs_df.shape)
display(final_selected_models_df.groupby("selection_track", dropna=False).size().reset_index(name="n_selected"))
display(guardrails_df)
display(selected_configs_df.head())


Images: 48
Objects: 1262
Final selected rows: (33, 30)
Restored selected configs: (33, 97)


,selection_track,n_selected
0,object_matrix_2way,2
1,object_matrix_3way,7
2,pixel_matrix_2way,3
3,pixel_matrix_3way,21


,check_name,passed,status,severity,details,n_records
0,final_selection_tracks_available,True,passed,error,[],33.0
1,selected_configs_restored,True,passed,error,,33.0
2,selected_configs_match_final_selection,True,passed,error,"{""restored"": 33, ""final_selected"": 33}",NaN
3,candidate_panel_available,True,passed,error,,1982.0
4,three_way_thresholds_available,True,passed,error,,32.0
5,train_batches_are_final_pure_batches,True,passed,error,"[1, 2, 3, 4]",NaN
6,projection_filters_select_mixtures,True,passed,error,"{""sample_kind"": [""mixture""]}",NaN
7,final_train_target_objects_available,True,passed,error,180 target objects,180.0
8,mixture_objects_available,True,passed,error,722 mixture objects,722.0


,selected_config_id,candidate_id_selected,selection_track,matrix_family_selected,decision_mode,metric_level,matrix_method_selected,preprocessing_selected,rule_for_refit_selected,n_components_selected,...,value_1,value_2,objective_fn_rate_max,objective_fp_rate_mean,objective_balanced_accuracy_mean,metric_equivalence_original_order,m,balanced_pixel_strategy,three_way_lower_threshold,three_way_upper_threshold
0,04C_refit_000306,simca_2f532685804685f4,object_matrix_2way,object_matrix,2way,object,object_median,absorbance_sg_d1,data_driven_emp_cv,3,...,0.090909,0.718696,0.471698,0.090909,0.718696,368,40.0,random,0.40,0.80
1,04C_refit_000290,simca_e6cd2d88651f1e6a,object_matrix_2way,object_matrix,2way,object,object_median,absorbance_sg_d1,data_driven_emp_cv,4,...,NaN,NaN,NaN,NaN,NaN,348,40.0,random,0.25,0.75
2,04C_refit_000430,simca_b237a6348e53cfc9,object_matrix_3way,object_matrix,3way,object,object_median,absorbance_sg_d2,simple_emp_cv,5,...,0.090909,0.614923,0.679245,0.090909,0.614923,507,40.0,random,0.35,0.70
3,04C_refit_000461,simca_c834a63e753e639e,object_matrix_3way,object_matrix,3way,object,object_median,absorbance_sg_d2,data_driven_emp_cv,5,...,0.109091,0.586964,0.716981,0.109091,0.586964,538,40.0,random,0.40,0.75
4,04C_refit_000597,simca_0e0daa82bb4f829f,object_matrix_3way,object_matrix,3way,object,object_median,absorbance_sg_d1,combined_index_chi2,6,...,0.036364,0.528988,0.905660,0.036364,0.528988,682,40.0,random,0.15,0.60


## Run Or Reload Mixture Application

When running, each selected model is refit on pure target objects from batches 1-4 and projected onto all mixture objects. The output metrics can be filtered to the track assigned in 06B.


In [4]:
mixture_outputs = {
    "final_selected_models": final_selected_models_df,
    "selected_configs": selected_configs_df,
    "guardrails": guardrails_df,
}

if RUN_MIXTURE_REFIT:
    mixture_outputs.update(
        run_mixture_application_batches(
            selected_configs_df=selected_configs_df,
            object_db=object_db,
            image_db=image_db,
            train_filters=MIXTURE_FINAL_TRAIN_FILTERS,
            projection_filters=MIXTURE_FILTERS,
            preprocessing_configs=preprocessing_configs_by_family,
            thresholds_df=selected_thresholds_df,
            evaluation_stage=EVALUATION_STAGE,
            wavelengths=wavelengths,
            random_state=RANDOM_STATE,
            replace=REPLACE_BALANCED_PIXELS,
            cv_n_splits=CV_N_SPLITS,
            cv_group_col=CV_GROUP_COL,
            target_class=TARGET_CLASS,
            non_target_label=NON_TARGET_LABEL,
            batch_size=MIXTURE_BATCH_SIZE,
            batch_metrics_dir=BATCH_METRICS_DIR,
            batch_objects_dir=BATCH_OBJECTS_DIR,
            batch_pixels_dir=BATCH_PIXELS_DIR,
            batch_3way_objects_dir=BATCH_3WAY_OBJECTS_DIR,
            save_batch_metric_tables=SAVE_BATCH_METRIC_TABLES,
            save_batch_object_tables=SAVE_BATCH_OBJECT_TABLES,
            save_batch_pixel_tables=SAVE_BATCH_PIXEL_TABLES,
            save_batch_3way_object_tables=SAVE_BATCH_3WAY_OBJECT_TABLES,
            save_combined_object_tables=SAVE_COMBINED_OBJECT_TABLES,
            save_combined_pixel_tables=SAVE_COMBINED_PIXEL_TABLES,
            save_combined_3way_object_tables=SAVE_COMBINED_3WAY_OBJECT_TABLES,
            keep_only_assigned_track_metrics=KEEP_ONLY_ASSIGNED_TRACK_METRICS,
            fixed_thresholds_path=THREE_WAY_THRESHOLDS_PATH,
        )
    )
else:
    if not USE_EXISTING_MIXTURE_OUTPUTS:
        raise RuntimeError("RUN_MIXTURE_REFIT is False and USE_EXISTING_MIXTURE_OUTPUTS is False.")
    missing_existing = missing_existing_mixture_paths(MIXTURE_PATHS)
    if missing_existing:
        raise FileNotFoundError(
            "RUN_MIXTURE_REFIT is False but required mixture outputs are missing:\n"
            + "\n".join(map(str, missing_existing))
        )
    mixture_outputs.update(load_existing_mixture_outputs(MIXTURE_PATHS))

mixture_outputs = validate_mixture_outputs(mixture_outputs)

protocol_df = build_mixture_protocol(
    {
        "notebook": "07_simca_mixture_application",
        "results_tag": RESULTS_TAG,
        "input_06b_dir": RESULTS_06B_DIR,
        "input_06a_dir": RESULTS_06A_DIR,
        "input_04c_dir": RESULTS_04C_DIR,
        "db_h5_path": DB_H5_PATH,
        "pca_selected_preprocessings_path": PCA_SELECTED_PREPROCESSINGS_PATH,
        "evaluation_stage": EVALUATION_STAGE,
        "target_class": TARGET_CLASS,
        "non_target_label": NON_TARGET_LABEL,
        "train_batches": MIXTURE_FINAL_TRAIN_BATCHES,
        "projection_filters": MIXTURE_FILTERS,
        "batch_size": MIXTURE_BATCH_SIZE,
        "run_mixture_refit": RUN_MIXTURE_REFIT,
        "use_existing_mixture_outputs": USE_EXISTING_MIXTURE_OUTPUTS,
        "keep_only_assigned_track_metrics": KEEP_ONLY_ASSIGNED_TRACK_METRICS,
        "save_combined_object_tables": SAVE_COMBINED_OBJECT_TABLES,
        "save_combined_pixel_tables": SAVE_COMBINED_PIXEL_TABLES,
        "save_combined_3way_object_tables": SAVE_COMBINED_3WAY_OBJECT_TABLES,
    },
    mixture_outputs,
)
mixture_outputs["protocol"] = protocol_df

saved_paths = save_mixture_outputs(
    mixture_outputs,
    MIXTURE_PATHS,
    save_combined_object_tables=SAVE_COMBINED_OBJECT_TABLES,
    save_combined_pixel_tables=SAVE_COMBINED_PIXEL_TABLES,
    save_combined_3way_object_tables=SAVE_COMBINED_3WAY_OBJECT_TABLES,
)

print("Saved:")
for path in saved_paths:
    print(" -", path)
display(protocol_df)


[mixture_application] batch_0001: candidates 1-10 / 33
[mixture_application] 04C_refit_000306


C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\decision\truth.py:126: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(


[mixture_application] 04C_refit_000290


C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\decision\truth.py:126: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(


[mixture_application] 04C_refit_000430


...

C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\decision\truth.py:126: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(


[mixture_application] 04C_refit_001157


C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\decision\truth.py:126: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(


Saved:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\07_simca_mixture_application_non_noisy_all\mixture_selected_configs.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\07_simca_mixture_application_non_noisy_all\mixture_2way_object_metrics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\07_simca_mixture_application_non_noisy_all\mixture_2way_pixel_metrics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\07_simca_mixture_application_non_noisy_all\mixture_3way_object_metrics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\07_simca_mixture_application_non_noisy_all\mixture_metrics_long.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\07_simca_mixture_application_non_noisy_all\mixture_object_diagnostics_by_image.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\07_simca_mixture_a

,notebook,results_tag,input_06b_dir,input_06a_dir,input_04c_dir,db_h5_path,pca_selected_preprocessings_path,evaluation_stage,target_class,non_target_label,...,n_selected_models,n_restored_configs,n_2way_object_metrics,n_2way_pixel_metrics,n_3way_object_metrics,n_metrics_long,n_object_image_diagnostics,n_pixel_image_diagnostics,n_3way_object_image_diagnostics,n_errors
0,07_simca_mixture_application,non_noisy_all,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,mixture_application,peanut,almond,...,33,33,5,5,28,38,100,100,560,0


## Compact Result Tables

These tables are the notebook-level inspection layer. A separate reporting script should generate the exhaustive figure set from the saved parquet outputs.


In [5]:
metrics_long_df = mixture_outputs["metrics_long"]
summary_df = mixture_outputs["summary"]
object_image_diagnostics_df = mixture_outputs.get("object_image_diagnostics", pd.DataFrame())
pixel_image_diagnostics_df = mixture_outputs.get("pixel_image_diagnostics", pd.DataFrame())
three_way_object_image_diagnostics_df = mixture_outputs.get("3way_object_image_diagnostics", pd.DataFrame())
objects_df = mixture_outputs.get("objects", pd.DataFrame())
pixels_df = mixture_outputs.get("pixels", pd.DataFrame())
objects_3way_df = mixture_outputs.get("3way_objects", pd.DataFrame())
errors_df = mixture_outputs.get("errors", pd.DataFrame())

display(summary_df)
display(
    metrics_long_df[
        [
            col for col in [
                "assigned_selection_track",
                "selection_track",
                "final_rank_in_track",
                "selected_config_id",
                "metric_level",
                "fn_rate",
                "fp_rate",
                "balanced_accuracy",
                "target_miss_rate",
                "non_target_false_accept_rate",
                "uncertain_rate",
                "coverage_rate",
                "decided_balanced_accuracy",
            ]
            if col in metrics_long_df.columns
        ]
    ].sort_values(
        [col for col in ["assigned_selection_track", "final_rank_in_track", "metric_level"] if col in metrics_long_df.columns]
    ).head(40)
)
display(errors_df.head())


,selection_track,assigned_selection_track,matrix_family,decision_mode,metric_level,n_models,n_rows,best_fn_rate,best_fp_rate,best_balanced_accuracy,median_fn_rate,median_fp_rate,median_balanced_accuracy,best_target_miss_rate,best_non_target_false_accept_rate,best_uncertain_rate,median_target_miss_rate,median_non_target_false_accept_rate,median_uncertain_rate
0,object_matrix_2way,object_matrix_2way,object_matrix,2way,object,2,2,0.321918,0.074653,0.787826,0.359589,0.088542,0.775935,NaN,NaN,NaN,NaN,NaN,NaN
1,object_matrix_2way,object_matrix_2way,object_matrix,2way,pixel,2,2,0.201086,0.373340,0.707185,0.219120,0.378942,0.700969,NaN,NaN,NaN,NaN,NaN,NaN
2,object_matrix_3way,object_matrix_3way,object_matrix,3way,object,7,7,0.000000,0.000000,0.971086,0.020548,0.015625,0.844444,0.0,0.000000,0.150970,0.020548,0.015625,0.319945
3,pixel_matrix_2way,pixel_matrix_2way,pixel_matrix,2way,object,3,3,0.000000,0.503472,0.744839,0.000000,0.769097,0.615451,NaN,NaN,NaN,NaN,NaN,NaN
4,pixel_matrix_2way,pixel_matrix_2way,pixel_matrix,2way,pixel,3,3,0.006981,0.675633,0.650112,0.007272,0.862434,0.565293,NaN,NaN,NaN,NaN,NaN,NaN
5,pixel_matrix_3way,pixel_matrix_3way,pixel_matrix,3way,object,21,21,0.000000,0.020833,0.986813,0.000000,0.644097,0.500000,0.0,0.020833,0.069252,0.000000,0.644097,0.288089


,assigned_selection_track,selection_track,final_rank_in_track,selected_config_id,metric_level,fn_rate,fp_rate,balanced_accuracy,target_miss_rate,non_target_false_accept_rate,uncertain_rate,coverage_rate,decided_balanced_accuracy
0,object_matrix_2way,object_matrix_2way,1,04C_refit_000306,object,0.321918,0.102431,0.787826,NaN,NaN,NaN,NaN,NaN
6,object_matrix_2way,object_matrix_2way,1,04C_refit_000306,pixel,0.201086,0.384545,0.707185,NaN,NaN,NaN,NaN,NaN
1,object_matrix_2way,object_matrix_2way,2,04C_refit_000290,object,0.397260,0.074653,0.764043,NaN,NaN,NaN,NaN,NaN
7,object_matrix_2way,object_matrix_2way,2,04C_refit_000290,pixel,0.237153,0.373340,0.694753,NaN,NaN,NaN,NaN,NaN
10,object_matrix_3way,object_matrix_3way,1,04C_refit_000430,object,0.000000,0.401042,0.651057,0.000000,0.401042,0.349030,0.650970,0.651057
11,object_matrix_3way,object_matrix_3way,2,04C_refit_000461,object,0.000000,0.338542,0.700000,0.000000,0.338542,0.362881,0.637119,0.700000
12,object_matrix_3way,object_matrix_3way,3,04C_refit_000597,object,0.006849,0.069444,0.930667,0.006849,0.069444,0.439058,0.560942,0.930667
13,object_matrix_3way,object_matrix_3way,4,04C_refit_000724,object,0.287671,0.000000,0.500000,0.287671,0.000000,0.150970,0.849030,0.500000
14,object_matrix_3way,object_matrix_3way,5,04C_refit_000726,object,0.027397,0.000000,0.953488,0.027397,0.000000,0.203601,0.796399,0.953488
15,object_matrix_3way,object_matrix_3way,6,04C_refit_000741,object,0.020548,0.015625,0.971086,0.020548,0.015625,0.319945,0.680055,0.971086


""


## Essential Figures

Only a few figures are shown here: model ranking, difficult images for one model, and one spatial diagnostic panel when pixel tables are available.


In [6]:
if len(metrics_long_df) > 0 and "balanced_accuracy" in metrics_long_df.columns:
    ranking_df = metrics_long_df.dropna(subset=["balanced_accuracy"]).copy()
    if len(ranking_df) > 0:
        fig = plot_model_metric_ranking(
            ranking_df,
            metric_col="balanced_accuracy",
            id_col="selected_config_id",
            family_col="assigned_selection_track" if "assigned_selection_track" in ranking_df.columns else "matrix_family",
            ascending=False,
            top_n=20,
            title="Mixture application - model ranking by balanced accuracy",
            show=False,
        )
        fig.show()

image_diag_for_plot = (
    three_way_object_image_diagnostics_df
    if len(three_way_object_image_diagnostics_df) > 0
    else object_image_diagnostics_df
)
if len(image_diag_for_plot) > 0:
    fig = plot_per_image_performance(
        image_diag_for_plot,
        image_col="source_image",
        metric_cols=("fn_rate", "fp_rate", "balanced_accuracy"),
        config_col="selected_config_id",
        sort_metric="fn_rate" if "fn_rate" in image_diag_for_plot.columns else "target_miss_rate",
        worst_first=True,
        top_n=20,
        title="Mixture application - difficult images",
        show=False,
    )
    fig.show()

diagnostic_config_id = None
diagnostic_image_key = None
if len(selected_configs_df) > 0:
    diagnostic_config_id = str(
        selected_configs_df.sort_values(
            [col for col in ["selection_track", "final_rank_in_track", "selected_config_id"] if col in selected_configs_df.columns]
        ).iloc[0]["selected_config_id"]
    )

diagnostic_images_df = choose_mixture_diagnostic_images(
    image_diag_for_plot,
    config_id=diagnostic_config_id,
    n_images=expcfg.MIXTURE_APPLICATION_DIAGNOSTIC_TOP_IMAGES,
)
if len(diagnostic_images_df) > 0:
    diagnostic_image_key = str(diagnostic_images_df.iloc[0]["source_image"])

display(diagnostic_images_df)

if diagnostic_config_id is not None and diagnostic_image_key is not None and len(objects_df) > 0 and len(pixels_df) > 0:
    diagnostic_object_df = objects_df[objects_df["selected_config_id"].astype(str).eq(diagnostic_config_id)].copy()
    diagnostic_pixel_df = pixels_df[pixels_df["selected_config_id"].astype(str).eq(diagnostic_config_id)].copy()
    radius = selected_configs_df.loc[
        selected_configs_df["selected_config_id"].astype(str).eq(diagnostic_config_id),
        "position_dilation_radius",
    ]
    radius = 3 if radius.empty or pd.isna(radius.iloc[0]) else int(radius.iloc[0])
    fig = plot_mixture_diagnostic_panel(
        image_key=diagnostic_image_key,
        image_db=image_db,
        object_db=object_db,
        object_df=diagnostic_object_df,
        pixel_df=diagnostic_pixel_df,
        target_class=TARGET_CLASS,
        dilation_radius=radius,
        crop_to_objects=True,
        padding=5,
        title=f"Mixture diagnostic - {diagnostic_config_id} - {diagnostic_image_key}",
        show=False,
    )
    fig.show()
else:
    print("Spatial diagnostic skipped: combined object/pixel tables are not available.")


""


Spatial diagnostic skipped: combined object/pixel tables are not available.


## Output Inventory


In [7]:
print("07_simca_mixture_application completed.")
display(list_result_files(RESULTS_DIR))


07_simca_mixture_application completed.


,file,suffixes,size_mb
0,mixture_pixels.parquet,.parquet,42.957077
1,mixture_3way_objects.parquet,.parquet,0.748947
2,mixture_objects.parquet,.parquet,0.640725
3,mixture_pixel_errors_by_image.parquet,.parquet,0.049756
4,mixture_3way_object_diagnostics_by_image.parquet,.parquet,0.039437
5,mixture_batches\metrics\batch_0001_2way_object...,.parquet,0.037951
6,mixture_batches\metrics\batch_0002_2way_object...,.parquet,0.037828
7,mixture_batches\metrics\batch_0003_2way_object...,.parquet,0.037814
8,mixture_metrics_long.parquet,.parquet,0.037171
9,mixture_batches\metrics\batch_0004_2way_object...,.parquet,0.034457
